# The effect of pitch on coherence

Notebook 3 asks whether the coherence spectrum moves with how fast the speech is.
This one asks the same question of the **high-frequency regime, 30-300 Hz**, with
**pitch (F0)** in place of the speech-unit rates.

The two axes line up exactly by construction.
`scripts/speech_units_pitch/formants_and_pitch.py` histograms each stimulus's voiced
F0 onto 2561 bins at 0.2 Hz spanning 0-512 Hz -- the *same* frequency grid the
coherence spectra are computed on. So a stimulus's pitch distribution can be drawn
straight onto its coherence spectrum with no interpolation or rebinning anywhere.

Ported from `exploration/4_pitch_vs_coh.ipynb`. The formatting follows notebook 3's
`speech_rate_and_coherence_<reference>` throughout -- same text-block width, same
gridspec proportions, same `scatter_panel`, same typography -- so the two read as a
pair.

## Figure produced

**`pitch_and_coherence_<reference>`**

* **(a)** one row per stimulus, ordered by median F0, standardised against that
  stimulus's own null exactly as notebook 3's panel (a) is.
* **(b)** stimuli pooled into 8 equal-duration groups by median F0, each group's
  pooled F0 distribution drawn under its coherence spectrum. That is the comparison
  the figure exists to make: does the coherence do anything where the voice is?
* **(c)**, **(d)** and **(e)** the fitted coherence peak against the group's median
  F0, in figure 2's standard bands: 30-60 Hz, 80-120 Hz (which overlaps the range the low
  voices occupy), and optionally 140-250 Hz -- roughly the first harmonic of that range.

**`pitch_peak_height_<reference>`** -- supplementary: the height of each fitted band
peak, rather than its frequency, against the group's median F0.

**`pitch_central_cluster_<reference>`** -- supplementary: peak frequency and peak
height again, on figure 2's 15-channel central cluster rather than all 64 sensors.

## Setup

`REFERENCE` selects the EEG reference; every cache and the figure name follow from it.
The input is the per-stimulus null files from `get_null_gammatone_coherence.py`
(`null_coherence_dir/gammatone_null_by_stimulus`) and the per-stimulus pitch
histograms from `scripts/speech_units_pitch/`.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D
from matplotlib.ticker import MaxNLocator
from scipy.ndimage import median_filter, uniform_filter1d
from scipy.stats import false_discovery_control, pearsonr
from tqdm.auto import tqdm

ROOT = Path.cwd().parent if Path.cwd().name == "figures" else Path.cwd()
CONFIG = json.loads((ROOT / "src" / "package_data" / "config.json").read_text())

REFERENCE = "avg"  # "avg" | "linked_mastoids" | "FCz"

sys.path.insert(0, str(ROOT / "src"))
from coherence.pipeline import resolve_reference_dirs

OUT_DIR = ROOT / "figures"
CACHE_ROOT = OUT_DIR / "figure_data"          # the tree notebooks 2 and 3 use


def cache_dir(reference):
    path = CACHE_ROOT / reference
    path.mkdir(parents=True, exist_ok=True)
    return path


STIMULUS_NULL_DIR = (resolve_reference_dirs(CONFIG, REFERENCE, ROOT)[1]
                     / "gammatone_null_by_stimulus")
PITCH_DIR = Path(CONFIG["speech_units_dir"]) / "pitch"

# --- house style, shared with figures 1-3 ------------------------------------
DEFAULT_FONTSIZE = 10
TICK_FONTSIZE = 8
SMALL_FONTSIZE = 6.5
PANEL_LABEL_FONTSIZE = 14
# Notebook 3's small-multiple sizes, so a scatter panel here is typeset exactly as
# one of its rows (c)-(d).
ROW_TITLE_FONTSIZE = 8
SCATTER_LABEL_FONTSIZE = 7.5
SCATTER_TICK_FONTSIZE = 7
CAPTION_FONTSIZE = 5

INK, MUTED, HAIRLINE = "#1b1b1a", "#6b6b68", "#d5d5d1"
NULL_FILL = "#b9b9b4"
MEDIAN_COLOUR = "#c1272d"
# Figure 2's pitch colour, so pitch reads as the same quantity across the notebooks.
PITCH_COLOUR = "#073b4c"

plt.rcParams.update({
    "font.size": TICK_FONTSIZE,
    "axes.labelsize": DEFAULT_FONTSIZE,
    "axes.titlesize": DEFAULT_FONTSIZE,
    "axes.edgecolor": INK,
    "axes.labelcolor": INK,
    "text.color": INK,
    "xtick.color": INK,
    "ytick.color": INK,
    "xtick.labelsize": TICK_FONTSIZE,
    "ytick.labelsize": TICK_FONTSIZE,
    "axes.linewidth": 0.8,
    "figure.dpi": 130,
    # TrueType, not matplotlib's default Type 3, which publishers flag.
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})


def bare(ax):
    """Recessive chrome: drop the top/right rules."""
    ax.spines[["top", "right"]].set_visible(False)
    return ax


BAND = (30.0, 300.0)
N_GROUPS = 8

# Only the first 100 of the 500 null draws are used. The null enters solely as a
# per-frequency MEDIAN, which stabilises long before 100 draws -- checked in the delta
# band, where 25, 50 and 100 gave identical results to four decimals -- and 500 draws
# over this band's 1351 bins would not fit in memory when pooled.
N_NULL_DRAWS = 100

# 50 Hz mains and its harmonics. Unlike notebook 3's 0.2-2 Hz band, this one runs
# straight through them. They are not removed -- the audio carries no mains, so a
# genuine envelope-EEG coherence should not sit there, and the null absorbs whatever
# does -- but they are kept out of the per-stimulus scaling, so a line artefact cannot
# set a stimulus's scale.
MAINS_HZ = np.arange(50.0, BAND[1] + 1, 50.0)
MAINS_HALF_WIDTH = 1.0

print(f"reference: {REFERENCE}  "
      f"({len(sorted(STIMULUS_NULL_DIR.glob('*.npz')))} stimulus null files)")

## Pitch

One histogram per stimulus, already on the coherence frequency grid. From it,
`median_f0` -- the 50th percentile of the voiced-frame distribution -- which both
orders panel (a) and forms panel (b)'s groups.

The extraction ran with a 30-370 Hz pitch floor and ceiling, so the distribution is
bounded by those settings rather than by the band drawn here.

In [ ]:
def percentile_from_histogram(counts, centres, fraction):
    """The frequency at which the cumulative count first reaches `fraction`."""
    cumulative = np.cumsum(counts) / counts.sum()
    return float(centres[np.searchsorted(cumulative, fraction)])


stimulus_names = sorted(path.name.replace("_gammatone_null.npz", "")
                        for path in STIMULUS_NULL_DIR.glob("*.npz"))

pitch_counts, median_f0 = [], []
for name in stimulus_names:
    with np.load(PITCH_DIR / f"{name}_pitch.npz") as saved:
        counts, pitch_centres = saved["counts"].astype(float), saved["bin_centres"]
    pitch_counts.append(counts)
    median_f0.append(percentile_from_histogram(counts, pitch_centres, 0.50))

pitch_counts = np.array(pitch_counts)              # (n_stimuli, 2561)
median_f0 = np.array(median_f0)

print(f"{len(stimulus_names)} stimuli, all with a pitch histogram")
print(f"median F0 across stimuli: {median_f0.min():.0f}-{median_f0.max():.0f} Hz "
      f"(median {np.median(median_f0):.0f})")

## Referencing each stimulus to its own null, and pooling into groups

Per stimulus, identical to notebook 3's panel (a) with the band moved up:

1. the stimulus's duration-weighted **complex-average** coherence, reduced to an
   **RMS magnitude** over the 64 sensors;
2. minus that stimulus's own **median null RMS, by frequency**;
3. divided by the **RMS of that centred trace**, one scalar per stimulus, computed
   with the mains bins excluded so a line artefact cannot set a stimulus's scale.

The same pass pools the stimuli into `N_GROUPS` blocks of as near to equal listening
time as the durations allow, by the same exact dynamic-programming partition notebook
3 uses. Pooling is on the **complex** coherence, duration-weighted, before any
magnitude is taken -- the order used everywhere in this project, and the reason a
group's spectrum is cleaner than its members'.

Each stimulus contributes its pitch distribution in proportion to **listening time**,
not to its own voiced-frame count: each histogram is normalised to unit area before
weighting. Otherwise a stimulus one participant heard would shape a group's drawn
distribution as strongly as one forty heard, while contributing forty times less to
the coherence trace it is drawn under. A group's quoted median is then the median *of
the distribution drawn*, so the fill and the rule cannot disagree.

In [ ]:
def form_groups(key, seconds, n_groups):
    """Stimuli sorted by ascending `key`, cut into exactly `n_groups` contiguous blocks
    of as near to equal listening time as the durations allow.

    Exact, by dynamic programming -- the same partition notebook 3 uses. A greedy
    accumulate-to-a-threshold walk cannot match it: it commits to early boundaries it
    can never revisit.
    """
    order = np.argsort(key)
    weights = seconds[order]
    n = len(weights)
    prefix = np.concatenate([[0.0], np.cumsum(weights)])
    target = prefix[-1] / n_groups

    cost = np.full((n + 1, n_groups + 1), np.inf)
    split = np.zeros((n + 1, n_groups + 1), dtype=int)
    cost[0, 0] = 0.0
    for k in range(1, n_groups + 1):
        for i in range(k, n - (n_groups - k) + 1):
            starts = np.arange(k - 1, i)
            candidate = cost[starts, k - 1] + (prefix[i] - prefix[starts] - target) ** 2
            best = int(np.argmin(candidate))
            cost[i, k] = candidate[best]
            split[i, k] = starts[best]

    groups, i = [], n
    for k in range(n_groups, 0, -1):
        j = split[i, k]
        groups.append(list(order[j:i]))
        i = j
    return groups[::-1]


def pool_pitch_groups(names, directory, in_band, groups, mains):
    """Per-stimulus traces and the pooled group spectra, in one pass over the files."""
    n_bins = int(in_band.sum())
    stimulus_rms = np.zeros((len(names), n_bins))
    stimulus_null_median = np.zeros_like(stimulus_rms)
    seconds = np.zeros(len(names))
    belongs = {index: k for k, members in enumerate(groups) for index in members}
    accumulators = [None] * len(groups)

    for index, name in enumerate(tqdm(names, desc="stimuli", unit="file")):
        with np.load(directory / f"{name}_gammatone_null.npz") as saved:
            weight = float(saved["total_analysed_duration"])
            real = saved["gammatone_coherence"].squeeze(1)[in_band]          # (F, ch)
            null = (saved["null_gammatone_coherence"][:N_NULL_DRAWS]
                    .squeeze(2)[:, in_band])                            # (D, F, ch)
        seconds[index] = weight
        stimulus_rms[index] = np.sqrt((np.abs(real) ** 2).mean(-1))
        stimulus_null_median[index] = np.median(
            np.sqrt((np.abs(null) ** 2).mean(-1)), axis=0)

        k = belongs[index]
        if accumulators[k] is None:
            accumulators[k] = [np.zeros(real.shape, np.complex128),
                               np.zeros(null.shape, np.complex64), 0.0]
        accumulators[k][0] += real * weight
        accumulators[k][1] += null * np.float32(weight)
        accumulators[k][2] += weight
        del real, null

    group_real_rms, group_null_median = [], []
    for k, (real_sum, null_sum, weight_sum) in enumerate(accumulators):
        group_real_rms.append(np.sqrt((np.abs(real_sum / weight_sum) ** 2).mean(-1)))
        power = np.abs(null_sum / np.complex64(weight_sum))
        power **= 2
        group_null_median.append(np.median(np.sqrt(power.mean(-1)), axis=0))
        del power
        accumulators[k] = None              # freed as we go: 8 x 69 MB otherwise

    # Step 3, with the mains bins kept out of the scale.
    centred = stimulus_rms - stimulus_null_median
    scale = np.sqrt((centred[:, ~mains] ** 2).mean(-1))
    return (centred / scale[:, None], stimulus_rms, stimulus_null_median, seconds,
            np.array(group_real_rms), np.array(group_null_median))


def pitch_groups(reference):
    """The cached per-stimulus reduction and group pooling for one reference.

    Keyed on the stimuli, the band, the group count and the draw count: every one of
    them changes the shape or the membership of what is stored, so a cache built under
    other values would load with the right shape and the wrong content.
    """
    directory = (resolve_reference_dirs(CONFIG, reference, ROOT)[1]
                 / "gammatone_null_by_stimulus")
    cache = (cache_dir(reference) /
             f"pitch_groups_{BAND[0]:g}-{BAND[1]:g}Hz_{N_GROUPS}groups"
             f"_{N_NULL_DRAWS}draws.npz")

    with np.load(directory / f"{stimulus_names[0]}_gammatone_null.npz") as first:
        axis = first["frequencies"]
    in_band = (axis >= BAND[0]) & (axis <= BAND[1])
    band_frequencies = axis[in_band]
    mains = np.any(np.abs(band_frequencies[:, None] - MAINS_HZ[None, :])
                   <= MAINS_HALF_WIDTH, axis=1)

    if cache.exists():
        with np.load(cache, allow_pickle=True) as saved:
            if [str(n) for n in saved["stimuli"]] == list(stimulus_names):
                return (band_frequencies, mains, saved["norm"], saved["rms"],
                        saved["null_median"], saved["seconds"],
                        saved["group_real_rms"], saved["group_null_median"],
                        [list(m) for m in saved["group_members"]])
        print(f"  {reference}: cached stimuli differ, re-pooling")

    # Durations first, so the groups are known before the single pass over the files.
    seconds = np.array([
        float(np.load(directory / f"{name}_gammatone_null.npz")["total_analysed_duration"])
        for name in tqdm(stimulus_names, desc="durations", unit="file")])
    groups = form_groups(median_f0, seconds, N_GROUPS)

    norm, rms, null_median, seconds, real_rms, group_null = pool_pitch_groups(
        stimulus_names, directory, in_band, groups, mains)
    np.savez(cache, frequencies=band_frequencies, norm=norm, rms=rms,
             null_median=null_median, seconds=seconds, group_real_rms=real_rms,
             group_null_median=group_null,
             group_members=np.array([np.array(m) for m in groups], dtype=object),
             stimuli=np.array(stimulus_names))
    return (band_frequencies, mains, norm, rms, null_median, seconds, real_rms,
            group_null, groups)


(frequencies, mains_mask, stimulus_norm, stimulus_rms, stimulus_null_median,
 stimulus_seconds, group_real_rms, group_null_median, group_members) = pitch_groups(REFERENCE)

group_hours = np.array([stimulus_seconds[m].sum() / 3600 for m in group_members])
group_pitch = np.array([
    (pitch_counts[m] / pitch_counts[m].sum(1, keepdims=True)
     * stimulus_seconds[m][:, None]).sum(0) for m in group_members])
group_median_f0 = np.array([percentile_from_histogram(h, pitch_centres, 0.50)
                            for h in group_pitch])

print(f"{N_GROUPS} groups by median F0; listening hours "
      f"{group_hours.min():.1f}-{group_hours.max():.1f} (sd {group_hours.std():.1f}), "
      f"stimuli per group {[len(m) for m in group_members]}")
print("group median F0 (of the listening-weighted pooled distribution): "
      + ", ".join(f"{v:.0f}" for v in group_median_f0) + " Hz")
print(f"null floor over {BAND[0]:g}-{BAND[1]:g} Hz: "
      f"{(group_null_median / group_real_rms).mean() * 100:.0f}% of the observed RMS")

In [ ]:
# --- locating the coherence peak -------------------------------------------------
# The same estimator notebook 3 uses: an order-4 polynomial fitted across the WHOLE
# band and solved for its interior maximum. Fitting every bin rather than a window
# around the argmax means there is no window parameter and no dependence on which
# single bin came out highest.
#
# Three bands, all figure 2's standard ones: 30-60 Hz (the low-gamma peak every group
# carries), 80-120 Hz (the mid-gamma peak, which overlaps the range the low voices
# occupy) and 140-250 Hz (roughly the first harmonic of that voice range). Earlier
# versions cut the first two wider, to 30-70 and 75-130 Hz, to fit the features of this
# spectrum; they now match the bands the rest of the paper uses. Every label naming them is
# derived from this dict rather than written out, so it cannot drift from what is fitted.
# Whether the third band is fitted at all -- ONE LINE. It is the most speculative of the
# three: 30-60 and 80-120 Hz are features this spectrum visibly has, whereas 140-250 Hz is
# where the first harmonic of the voice range would land if it were tracked, and it is the
# band carrying three mains harmonics rather than one. Set it False and the band leaves the
# fits, panel (b)'s overlay, the FDR family and the scatter row in one move -- the row then
# draws two panels across the same width, so nothing else needs touching.
FIT_HIGH_BAND = False

ALL_FIT_BANDS = {"low": (30.0, 60.0), "pitch": (80.0, 120.0), "high": (140.0, 250.0)}
FIT_BANDS = {key: span for key, span in ALL_FIT_BANDS.items()
             if FIT_HIGH_BAND or key != "high"}
FIT_MARKERS = {"low": "s", "pitch": "o", "high": "^"}
FIT_LABELS = {key: f"Fit + peak, {lo:g}-{hi:g} Hz"
              for key, (lo, hi) in FIT_BANDS.items()}
POLYNOMIAL_ORDER = 4
fit_within = {key: (frequencies >= lo) & (frequencies <= hi)
              for key, (lo, hi) in FIT_BANDS.items()}


def band_polynomial_fit(band_frequencies, values, degree=POLYNOMIAL_ORDER, span=None):
    """Least-squares polynomial across a whole band. Returns (coefficients, span).

    The band is mapped onto [-1, 1] before fitting -- a polynomial in raw Hz over
    these ranges is badly conditioned.

    `span` fixes that mapping to the band's own edges. It has to be given whenever
    bins are dropped from the middle of a band, or the domain would be rescaled to
    whatever bins happened to survive and the fit would no longer be over the band.
    """
    low, high = (band_frequencies[0], band_frequencies[-1]) if span is None else span
    scaled = 2 * (band_frequencies - low) / (high - low) - 1
    return np.polyfit(scaled, values, degree), (low, high)


def evaluate_band_fit(coefficients, span, frequencies_out):
    """The fitted curve at arbitrary frequencies inside the band."""
    low, high = span
    return np.polyval(coefficients, 2 * (frequencies_out - low) / (high - low) - 1)


# The mains harmonics are left out of every fit. There is exactly one in each band --
# 50 Hz in the upper part of 30-60, 100 Hz in the middle of 80-120 -- so the one line
# artefact each band carries sits exactly where that band's peak is looked for. A peak
# must not be allowed to BE an artefact.
#
# Measured on the earlier, wider bands, it barely moved the answer: an order-4
# polynomial over ~200 bins has little leverage from the 11 that are masked. That is a
# reassurance, not a reason to leave them in. The 140-250 Hz band carries
# three harmonics (150, 200, 250), so it loses 28 of its 551 bins.
EXCLUDE_MAINS_FROM_FITS = True

fit_keep = {key: (within & ~mains_mask) if EXCLUDE_MAINS_FROM_FITS else within
            for key, within in fit_within.items()}
# The band's own edges, kept separately: dropping an interior notch must not rescale
# the polynomial's domain.
fit_span = {key: (frequencies[within][0], frequencies[within][-1])
            for key, within in fit_within.items()}


def fit_band(values, key, degree=POLYNOMIAL_ORDER):
    """(coefficients, span) for one band of one row, mains notches left out.

    The single place a band fit is set up, so the fit drawn in panel (b) and the peak
    reported in (c)-(d) cannot be computed over different bins.
    """
    keep = fit_keep[key]
    return band_polynomial_fit(frequencies[keep], values[keep], degree,
                               span=fit_span[key])


def band_polynomial_peak(values, key, degree=POLYNOMIAL_ORDER):
    """Maximum of the fitted polynomial, its R2, and whether the maximum is interior.

    Falls back to the fit's best kept bin where there is no interior maximum -- which
    the caller is told about rather than left to infer from a suspicious frequency.
    """
    coefficients, span = fit_band(values, key, degree)
    kept_frequencies = frequencies[fit_keep[key]]
    observed = values[fit_keep[key]]
    fitted = evaluate_band_fit(coefficients, span, kept_frequencies)
    r2 = 1 - (((observed - fitted) ** 2).sum()
              / ((observed - observed.mean()) ** 2).sum())

    roots = np.roots(np.polyder(coefficients))
    roots = roots[np.abs(roots.imag) < 1e-9].real
    curvature = np.polyval(np.polyder(np.polyder(coefficients)), roots)
    inside = roots[(curvature < 0) & (roots >= -1) & (roots <= 1)]
    if inside.size == 0:
        return kept_frequencies[np.argmax(fitted)], r2, False
    low, high = span
    best = inside[np.argmax(np.polyval(coefficients, inside))]
    return low + (best + 1) * (high - low) / 2, r2, True


# Fitted on the unsmoothed excess, as notebook 3's band fits are.
group_excess = group_real_rms - group_null_median
group_peak, group_peak_r2, group_peak_interior = {}, {}, {}
for key, within in fit_within.items():
    results = [band_polynomial_peak(group_excess[k], key) for k in range(N_GROUPS)]
    group_peak[key] = np.array([peak for peak, _, _ in results])
    group_peak_r2[key] = np.array([r2 for _, r2, _ in results])
    group_peak_interior[key] = [interior for _, _, interior in results]

if not FIT_HIGH_BAND:
    print(f"140-250 Hz band off: fitting {len(FIT_BANDS)} bands, "
          f"panel (e) is not drawn (FIT_HIGH_BAND)")
for key, (low, high) in FIT_BANDS.items():
    edge = sum(not interior for interior in group_peak_interior[key])
    print(f"order-{POLYNOMIAL_ORDER} fit over {low:g}-{high:g} Hz "
          f"({int(fit_keep[key].sum())} of {int(fit_within[key].sum())} bins, "
          f"mains excluded): peak "
          f"{group_peak[key].mean():.1f} Hz (sd {group_peak[key].std():.1f}), "
          f"R2 {group_peak_r2[key].mean():.2f}"
          + (f", {edge} of {N_GROUPS} pinned to a band edge" if edge else ""))

## The figure

Laid out as notebook 3's `speech_rate_and_coherence_<reference>`: the same 6.27 in
text-block width, the same gridspec proportions -- a top row of two large panels at
2.8 in and a scatter row at 1.1 in -- and the same `scatter_panel`, so a panel here
is typeset exactly as one of that figure's.

Panel (b)'s traces are lightly smoothed for display. 1351 bins across 2.5 in of panel
is far past print resolution, so the unsmoothed line is ink rather than information.
The 9-bin kernel is 1.8 Hz, one to two orders of magnitude narrower than the F0
distributions being compared against, so it cannot manufacture an alignment -- and the
fits are computed on the unsmoothed excess regardless.

The pitch histograms are likewise prepared for display only: zeroed outside
50-350 Hz, then median-filtered over 11 Hz. Every number reported, `group_median_f0`
included, stays on the untouched distribution, and the cell prints how far the two
differ.

In [ ]:
# Taller than the 6.3 the two-row layout needs, because hspace below is opened up to
# put air between the key and the scatter row; the extra height keeps the panels the
# size they were rather than letting the wider gutter eat them.
FIGSIZE = (6.27, 5.7)
OFFSET = 1.05

# Extra y range for the stack, as a fraction of its own height, split top and bottom.
# The rows are drawn baseline-to-baseline, so without it the top trace runs into the
# panel edge and the bottom one sits on the axis.
STACK_YLIM_MARGIN = 0.05

# Notebook 3's scatter constants, verbatim.
MARKER_RADIUS_PER_HOUR = 0.28
AXIS_MARGIN = 0.14

# Ticks fixed rather than left to a locator, so the two panels share one x ladder and
# each y ladder is even. The locator picks its own round numbers from each panel's own
# range, which left the two on different ladders.
SCATTER_XTICKS = [100, 150, 200, 250]
SCATTER_YTICKS = {"low": [40, 45, 50, 55], "pitch": [85, 90, 95, 100, 105, 110],
                  "high": [160, 170, 180]}


def tick_limits(values, ticks, margin=AXIS_MARGIN):
    """Limits covering both the data and every requested tick, with the usual pad.

    Either can bind: the lowest group sits at 93 Hz, below the first x tick, while the
    highest y ticks sit above every peak. The union keeps every tick on the axis
    without ever clipping a marker.
    """
    low = min(float(np.min(values)), min(ticks))
    high = max(float(np.max(values)), max(ticks))
    pad = margin * (high - low)
    return low - pad, high + pad

# Boxcar widths in bins of the 0.2 Hz grid, for DISPLAY ONLY. Neither touches the
# data the fits are computed on, which is always the unsmoothed excess.
#
# (a) defaults to 1, no smoothing: the plate is per stimulus, and a stimulus's own
# noise is part of what it shows.
#
# (b) needs its own. At 1 the rows are 6.7x rougher bin to bin, and the mains spikes --
# 1.3x the largest real feature in the band -- go from being spread out to standing
# above everything else in the panel. 9 bins is 1.8 Hz, one to two orders of magnitude
# narrower than the F0 distributions being compared against, so it cannot manufacture
# an alignment. Set it to 1 to see the raw rows.
HEATMAP_SMOOTH_BINS = 5
STACK_SMOOTH_BINS = 5

# Per-row caption in panel (b), the same device notebook 3's supplementary group figure
# uses: placed at the row's own baseline so it travels with the row, rather than collected
# into one block per panel. What a row is made of belongs beside the row.
#
# 5 pt is not a house size -- it is what fits the line into the empty right-hand end of a row
# without reaching back into the fitted curves. Right-aligned to the axis for the same reason:
# the left of every row is where the 30-60 and 80-120 Hz fits are drawn.
ROW_CAPTIONS = True
ROW_CAPTION_FONTSIZE = 5
ROW_CAPTION_Y = 1.0             # of the row height, from its baseline; `va="top"` hangs down
# The median-F0 rule reaches four of the eight captions where it runs the full height of its
# row, so with captions on it stops below them. It marks an x position; its height carries
# nothing, so it is the element that gives way rather than the text being moved off the row.
ROW_RULE_TOP = 0.66 if ROW_CAPTIONS else 1.0

# Pitch histograms, prepared FOR DISPLAY ONLY.
PITCH_DISPLAY_RANGE = (50.0, 350.0)
PITCH_MEDIAN_FILTER_HZ = 11.0

# Panel (b) row scaling -- ONE LINE TO REVERT, and the same choice notebook 3's
# stacked panel makes.
#   True : every row divided by the SAME number, so a row's height is its coherence
#          and rows are comparable. The cost is that the 40-50 Hz shoulder, which sets
#          the global maximum, squashes the rest.
#   False: each row divided by its own maximum. Every row fills its band, but the
#          heights mean nothing.
COMMON_ROW_SCALE = True

# What panel (b) DRAWS -- ONE LINE.
#   True : the RMS coherence as measured, floor included, which is what notebook 3's
#          stacked panel shows. A row's flat stretch then sits at that group's own
#          noise floor rather than at zero, and the floor is part of what the eye
#          compares between rows.
#   False: the null-corrected excess (measured - null median). The legend label in
#          panel (b) should say so if this is switched back.
# The fits, and every number in (c) and (d), are on the excess either way. When the
# measured spectrum is drawn the fitted curve is lifted by the floor so it still lands
# on the trace it belongs to.
DRAW_MEASURED = True


def scatter_panel(ax, x, y, hours, r, p, *, xlabel=None, ylabel=None, headroom=0.0,
                  corner="upper left", xticks=None, yticks=None):
    """One group-level scatter, styled exactly as notebook 3's rows (c)-(d)."""
    ax.scatter(x, y, s=(MARKER_RADIUS_PER_HOUR * hours) ** 2, color=INK, alpha=0.55,
               linewidths=0, zorder=2)
    slope, intercept = np.polyfit(x, y, 1)
    span = np.linspace(x.min(), x.max(), 2)
    ax.plot(span, slope * span + intercept, color=MEDIAN_COLOUR, lw=1.1, zorder=3)
    ax.margins(x=AXIS_MARGIN, y=AXIS_MARGIN)
    if headroom and yticks is None:
        # An explicit limit overrides the y margin, so the padding is reapplied here.
        low, high = y.min(), y.max()
        ax.set_ylim(low - AXIS_MARGIN * (high - low), high + headroom * (high - low))
    x_anchor, align = (0.05, "left") if corner == "upper left" else (0.95, "right")
    ax.text(x_anchor, 0.95, f"$r$ = {r:+.2f}\n$p_{{FDR}}$ = {p:.3f}",
            transform=ax.transAxes, va="top", ha=align,
            fontsize=SCATTER_TICK_FONTSIZE, color=INK if p < 0.05 else MUTED)
    if xlabel:
        ax.set_xlabel(xlabel, fontsize=SCATTER_LABEL_FONTSIZE)
    if ylabel:
        ax.set_ylabel(ylabel, fontsize=SCATTER_LABEL_FONTSIZE)
    # Fixed ticks where they are given, a locator where they are not -- setting ticks
    # replaces the locator outright, so the two cannot both be applied.
    if xticks is None:
        ax.xaxis.set_major_locator(MaxNLocator(4))
    else:
        ax.set_xlim(*tick_limits(x, xticks))
        ax.set_xticks(xticks)
    if yticks is None:
        ax.yaxis.set_major_locator(MaxNLocator(4))
    else:
        ax.set_ylim(*tick_limits(y, yticks))
        ax.set_yticks(yticks)
    ax.tick_params(labelsize=SCATTER_TICK_FONTSIZE)
    bare(ax)


# --- display transforms ----------------------------------------------------------
pitch_resolution = float(np.diff(pitch_centres).mean())
pitch_filter_bins = int(round(PITCH_MEDIAN_FILTER_HZ / pitch_resolution))
outside = ((pitch_centres < PITCH_DISPLAY_RANGE[0])
           | (pitch_centres > PITCH_DISPLAY_RANGE[1]))
group_pitch_display = group_pitch.copy()
group_pitch_display[:, outside] = 0.0
group_pitch_display = np.array([median_filter(h, size=pitch_filter_bins)
                                for h in group_pitch_display])

dropped = group_pitch[:, outside].sum(1) / group_pitch.sum(1)
shifted = np.abs(np.array([percentile_from_histogram(h, pitch_centres, 0.50)
                           for h in group_pitch_display]) - group_median_f0)
print(f"display transform: {pitch_filter_bins}-bin ({PITCH_MEDIAN_FILTER_HZ:g} Hz) "
      f"median filter, zeroed outside {PITCH_DISPLAY_RANGE[0]:g}-"
      f"{PITCH_DISPLAY_RANGE[1]:g} Hz")
print(f"  mass removed per group {dropped.min() * 100:.2f}-{dropped.max() * 100:.2f}%; "
      f"median moves {shifted.min():.1f}-{shifted.max():.1f} Hz "
      f"(every rule and number stays on the unfiltered distribution)")

in_band_pitch = (pitch_centres >= BAND[0]) & (pitch_centres <= BAND[1])
# The mains notches are filled by interpolation BEFORE the boxcar runs, not masked
# after it. A 9-bin kernel spreads a spike over +-0.8 Hz, so smoothing first and
# hiding the mains bins afterwards would leave the contamination sitting just outside
# the mask. Display only -- the fits have their own exclusion, on unsmoothed data.
INTERPOLATE_MAINS_FOR_DISPLAY = True


def fill_mains(values):
    """Linear interpolation across the mains notches, row by row."""
    filled = np.array(values, float, copy=True)
    good = ~mains_mask
    for row in filled:
        row[mains_mask] = np.interp(frequencies[mains_mask], frequencies[good],
                                    row[good])
    return filled


group_drawn = group_real_rms if DRAW_MEASURED else group_excess
if INTERPOLATE_MAINS_FOR_DISPLAY:
    group_drawn = fill_mains(group_drawn)
if STACK_SMOOTH_BINS > 1:
    group_drawn = uniform_filter1d(group_drawn, STACK_SMOOTH_BINS, axis=-1,
                                   mode="nearest")
# Where a row's baseline sits. Mapping [0, max] onto the row leaves every trace
# hovering in the top third, because with the measured spectrum drawn the floor is
# ~73% of the panel maximum and the two thirds beneath it are empty pedestal. The
# lower limit is lifted to the lowest value the rows actually reach, so a profile
# fills its slot.
#
# Under COMMON_ROW_SCALE this is ONE affine map shared by every row, so rows stay
# comparable with each other -- it changes where the axis starts, not the data. What
# it is not is a per-row or per-frequency correction; the baseline rule now marks that
# common lower limit rather than zero.
ROW_BOTTOM_MARGIN = 0.05        # of the drawn range, so nothing sits on the rule

# The mains bins are kept out of whichever scale is in force -- a line artefact must
# not be what sets a row's height or its floor.
if COMMON_ROW_SCALE:
    drawn_low = np.full(N_GROUPS, group_drawn[:, ~mains_mask].min())
    drawn_high = np.full(N_GROUPS, group_drawn[:, ~mains_mask].max())
else:
    drawn_low = group_drawn[:, ~mains_mask].min(axis=1)
    drawn_high = group_drawn[:, ~mains_mask].max(axis=1)
row_base = drawn_low - ROW_BOTTOM_MARGIN * (drawn_high - drawn_low)
row_scale = drawn_high - row_base


def to_row(values, height):
    """Drawn values mapped onto their row, from the common baseline upwards."""
    return (values - row_base[height]) / row_scale[height]


def fit_lift(frequencies_out, height):
    """What to add to a drawn fit so it lands on the row's drawn trace.

    The fits are of the excess; when the row draws the measured spectrum the fit has
    to be put back on top of that row's floor. Interpolated, because a fit is drawn on
    a dense grid rather than on the 0.2 Hz bins.
    """
    if not DRAW_MEASURED:
        return 0.0
    return np.interp(frequencies_out, frequencies, group_null_median[height])

# --- the figure ------------------------------------------------------------------
fig = plt.figure(figsize=FIGSIZE)
# Notebook 3's proportions: a top row of two large panels and one scatter row, with
# the same hspace and wspace it settled on. wspace is a fraction of the average
# column, and these columns are twice its width, so the value is halved to leave the
# same gutter in inches.
# Six columns: the top pair spans three each, the three scatters two each, so both
# rows' panels start and end on the same grid lines. wspace is a fraction of the
# AVERAGE column, and a column is now a sixth of the width rather than a half, so the
# value rises to leave the same gutter in inches.
outer = fig.add_gridspec(2, 6, height_ratios=[2.5, 1.15], hspace=0.62, wspace=0.55)

# --- (a) one row per stimulus, ordered by median F0 -------------------------------
ax_heat = fig.add_subplot(outer[0, 0:3])
row_order = np.argsort(median_f0)
plate = stimulus_norm[row_order]
if HEATMAP_SMOOTH_BINS > 1:
    # mode="nearest" rather than a zero-padded convolution: padding with zeros dims
    # the first and last bins of every row, and both edges of this band are plotted.
    plate = uniform_filter1d(plate, HEATMAP_SMOOTH_BINS, axis=-1, mode="nearest")
# Clipped to the bulk AFTER any smoothing, since these are limits for what is drawn:
# the most extreme cells reach well past the 99th percentile and on an autoscale would
# take most of the colour range.
vmin, vmax = np.percentile(plate, [1, 99])
ax_heat.pcolormesh(frequencies, np.arange(len(plate)), plate, shading="auto",
                   cmap="cividis", rasterized=True, vmin=vmin, vmax=vmax)
# One marker per stimulus at its median F0 -- the same encoding notebook 3's panel (a)
# uses for the unit rates.
ax_heat.scatter(median_f0[row_order], np.arange(len(plate)), s=6, c=PITCH_COLOUR,
                marker="o", linewidths=0.6, edgecolors="w", zorder=3,
                label="Stimulus median F0")
ax_heat.set_xlim(*BAND)
ax_heat.set_yticks([])
ax_heat.set_xlabel("Frequency (Hz)", fontsize=SCATTER_LABEL_FONTSIZE)
ax_heat.set_ylabel("Stimulus\n(ordered by median F0)", fontsize=SCATTER_LABEL_FONTSIZE)
ax_heat.tick_params(labelsize=SCATTER_TICK_FONTSIZE)
ax_heat.set_title("RMS coherence for individual stimuli", fontsize=ROW_TITLE_FONTSIZE)
# In the plate rather than in the key below, as notebook 3's panel (a) does. Upper
# left is the free corner here: the rows are ordered by F0, so the markers run from
# the bottom left to the top right.
ax_heat.legend(loc="lower right", fontsize=6, frameon=True, facecolor="white",
               framealpha=0.85, edgecolor="none", handletextpad=0.2,
               labelspacing=0.25, borderpad=0.35, markerscale=1.4)

# --- (b) groups by median F0, pooled ---------------------------------------------
ax_stack = fig.add_subplot(outer[0, 3:6])
for height, members in enumerate(group_members):
    baseline = height * OFFSET
    ax_stack.axhline(baseline, color=HAIRLINE, lw=0.4, zorder=1)

    # The group's pooled pitch distribution, on the same axis, scaled to the row.
    # Pitch is a different quantity with its own units, so it keeps filling the row
    # whatever the coherence scaling is -- it is a reference, not a measurement.
    pooled = group_pitch_display[height][in_band_pitch]
    if pooled.max() > 0:
        ax_stack.fill_between(frequencies, baseline, baseline + pooled / pooled.max(),
                              color=PITCH_COLOUR, alpha=0.30, lw=0, zorder=2)
    ax_stack.plot(frequencies, baseline + to_row(group_drawn[height], height),
                  color=INK, lw=0.7, zorder=4)

    # Both fits and their maxima, computed on the unsmoothed excess, drawn on the
    # row's scale.
    for key, (fit_low, fit_high) in FIT_BANDS.items():
        within = fit_within[key]
        coefficients, span = fit_band(group_excess[height], key)
        dense = np.linspace(fit_low, fit_high, 200)
        ax_stack.plot(dense,
                      baseline + to_row(evaluate_band_fit(coefficients, span, dense)
                                        + fit_lift(dense, height), height),
                      color=MEDIAN_COLOUR, lw=1.0, alpha=0.9, zorder=5)
        peak = group_peak[key][height]
        ax_stack.plot(peak,
                      baseline + to_row(evaluate_band_fit(coefficients, span, peak)
                                        + fit_lift(peak, height), height),
                      FIT_MARKERS[key], color=MEDIAN_COLOUR, markersize=4,
                      markeredgecolor="white", markeredgewidth=0.5, zorder=6)

    ax_stack.plot([group_median_f0[height]] * 2, [baseline, baseline + ROW_RULE_TOP],
                  color=PITCH_COLOUR, lw=1.2, solid_capstyle="butt", zorder=7)

    # Above the F0 fill it sometimes sits over, which is pale enough to read through.
    if ROW_CAPTIONS:
        ax_stack.text(BAND[1], baseline + ROW_CAPTION_Y,
                      f"LH={group_hours[height]:.1f}; "
                      f"NS={len(members)}; "
                      f"Median F0={group_median_f0[height]:.0f}",
                      fontsize=ROW_CAPTION_FONTSIZE, va="top", ha="right", color=INK,
                      zorder=8)

ax_stack.set_xlim(*BAND)
stack_low, stack_high = -0.08, (N_GROUPS - 1) * OFFSET + 1.12
stack_pad = STACK_YLIM_MARGIN * (stack_high - stack_low) / 2
ax_stack.set_ylim(stack_low - stack_pad, stack_high + stack_pad)
ax_stack.set_yticks([])
ax_stack.set_xlabel("Frequency (Hz)", fontsize=SCATTER_LABEL_FONTSIZE)
ax_stack.tick_params(axis="x", labelsize=SCATTER_TICK_FONTSIZE)
ax_stack.set_title("RMS coherence for\nstimuli grouped by median F0",
                   fontsize=ROW_TITLE_FONTSIZE)
for spine in ("top", "right", "left"):
    ax_stack.spines[spine].set_visible(False)

# --- (c) and (d): each fitted peak against the group's pitch ---------------------
# Titles derived from FIT_BANDS rather than written out, so a band cannot be renamed
# without its title following.
SCATTER_TITLES = {key: f"{low:g}-{high:g} Hz peak"
                  for key, (low, high) in FIT_BANDS.items()}

# One BH family over the scatter panels: they are the same question asked of each
# band, so correcting them together is the counterpart of notebook 3's within-row
# family. Both uncorrected and corrected values are printed.
raw_p, statistics = {}, {}
for key in FIT_BANDS:
    result = pearsonr(group_median_f0, group_peak[key])
    statistics[key], raw_p[key] = result.statistic, result.pvalue
corrected = dict(zip(FIT_BANDS, false_discovery_control(
    np.array([raw_p[key] for key in FIT_BANDS]), method="bh")))

# The six columns are split evenly between however many bands are fitted -- two columns
# each for three bands, three each for two -- so the row spans exactly the width of (a) and
# (b) above it whatever FIT_HIGH_BAND is set to, and its panels start and end on grid lines.
SCATTER_WIDTH, remainder = divmod(outer.ncols, len(FIT_BANDS))
assert remainder == 0, f"{len(FIT_BANDS)} bands do not divide {outer.ncols} columns evenly"

scatter_axes = []
for column, key in enumerate(FIT_BANDS):
    ax = fig.add_subplot(outer[1, column * SCATTER_WIDTH:(column + 1) * SCATTER_WIDTH])
    scatter_axes.append(ax)
    low, high = FIT_BANDS[key]
    scatter_panel(ax, group_median_f0, group_peak[key], group_hours,
                  statistics[key], corrected[key],
                  xlabel="Group median F0 (Hz)",
                  # Only the leftmost is labelled: all three say the same thing, and
                  # at three across the width the repeats cost more than they say.
                  ylabel="Peak of RMS coherence (Hz)" if column == 0 else None,
                  xticks=SCATTER_XTICKS, yticks=SCATTER_YTICKS[key])
    # A title per panel rather than notebook 3's one-title-per-row: there the four
    # panels of a row share a measure and differ only in which rate is on x, so one
    # title covers them; here the two panels are different measures. Kept short -- the
    # band is already on the y axis, so the title carries what the band IS. A centred
    # row title at this width ran straight through the (d) marker.
    ax.set_title(SCATTER_TITLES[key], fontsize=ROW_TITLE_FONTSIZE)

# Per-panel x offsets, as in notebook 3: (a) clears its two-line y-label and (b)
# carries nothing on the left.
# Panel (b)'s key, under the (a)/(b) row rather than inside (b). In the panel it
# covered the lower rows -- five entries need more height than one row has, and with
# the baseline lifted the traces now reach into the corner it was sitting in. Hung
# from the bottom of that row, clear of the x-labels, so it follows the height ratios.
#
# Panel (b)'s six entries only -- (a)'s marker is named inside (a). Two columns of
# three: matplotlib fills column by column, top to bottom, so the first column takes
# what is drawn per row and the second the three band fits. Centred under (b) rather
# than under the figure, since that is the panel it belongs to.
stack_box = ax_stack.get_position()
fig.legend(handles=[
    Line2D([0], [0], color=INK, lw=0.8, label="Coherence spectrum"),
    Line2D([0], [0], color=PITCH_COLOUR, lw=4, alpha=0.4,
           label="Pooled F0 distribution"),
    Line2D([0], [0], color=PITCH_COLOUR, lw=1.2, label="Group median F0"),
] + [Line2D([0], [0], color=MEDIAN_COLOUR, lw=1.0, marker=FIT_MARKERS[key],
            markersize=4, label=FIT_LABELS[key]) for key in FIT_BANDS],
    loc="upper center",
    bbox_to_anchor=((stack_box.x0 + stack_box.x1) / 2,
                    min(ax_heat.get_position().y0, stack_box.y0) - 0.05),
    ncol=2, fontsize=6, frameon=False, handlelength=1.8, labelspacing=0.3,
    columnspacing=1.6)

for ax, label, offset in [(ax_heat, "a)", -0.18), (ax_stack, "b)", -0.06)]:
    ax.text(offset, 1.08, label, transform=ax.transAxes,
            fontsize=PANEL_LABEL_FONTSIZE, fontweight="bold", va="bottom", ha="left")

# The scatter markers on their own axes, clear of their titles: each title is short and
# centred, so the marker sits well left of where it starts. The offset is in axes fractions,
# so it is scaled by the panel width -- otherwise dropping to two wider panels would push the
# markers further out in inches than the (a)/(b) markers above them.
SCATTER_LABEL_X = -0.15 * 2 / SCATTER_WIDTH
for ax, label in zip(scatter_axes, ("c)", "d)", "e)")):
    ax.text(SCATTER_LABEL_X, 1.08, label, transform=ax.transAxes,
            fontsize=PANEL_LABEL_FONTSIZE, fontweight="bold", va="bottom", ha="left")

stem = f"pitch_and_coherence_{REFERENCE}"
fig.savefig(OUT_DIR / f"{stem}.pdf", bbox_inches="tight", pad_inches=0.3)
fig.savefig(OUT_DIR / f"{stem}.png", bbox_inches="tight", pad_inches=0.3, dpi=200)
plt.show()

# The per-group numbers panel (b) used to caption. Off the figure now, so they are
# printed here to be set as a table in the document.
print(f"\n{'group':>5s} {'median F0 (Hz)':>15s} {'listening h':>12s} {'stimuli':>8s}"
      + "".join(f"{f'peak {low:g}-{high:g}':>14s}"
                for low, high in FIT_BANDS.values()))
for height in range(N_GROUPS):
    print(f"{height:5d} {group_median_f0[height]:15.0f} {group_hours[height]:12.1f} "
          f"{len(group_members[height]):8d}"
          + "".join(f"{group_peak[key][height]:14.1f}" for key in FIT_BANDS))

print()
for key, (low, high) in FIT_BANDS.items():
    print(f"{low:g}-{high:g} Hz peak vs group median F0: r = {statistics[key]:+.2f}, "
          f"p = {raw_p[key]:.3f}, p_FDR = {corrected[key]:.3f}  (n = {N_GROUPS})")
print(f"p_FDR: Benjamini-Hochberg over the {len(FIT_BANDS)} panels, which ask the "
      f"same question of {len(FIT_BANDS)} bands.")

## Supplementary figure -- the height of each band's peak

Panels (c)-(d) ask *where* in a band the coherence peaks. This asks *how strong* it is
there, the counterpart of notebook 3's peak-height figure: each group's band fit,
evaluated at its own fitted peak, against the group's median F0. The fits are the ones
panel (b) draws and (c)-(d) locate peaks with -- on the null-corrected excess, mains
notches left out -- so the height is the y-coordinate of the very point whose
x-coordinate those panels correlate.

The same BH family over the fitted bands as (c)-(d). The cell also repeats every test
on the measured spectrum, floor included, evaluated at the same peak frequency: the
groups' listening times are near-equal (15-19 h), so the floor should barely move them,
and the printout says whether it does.

In [ ]:
# The fit's value at its own peak, per group and band. Excess: the same fit panel (b)
# draws. Measured: the measured RMS fitted over the same bins and read at the same peak.
group_height, group_height_measured = {}, {}
for key in FIT_BANDS:
    group_height[key] = np.array([
        float(evaluate_band_fit(*fit_band(group_excess[k], key), group_peak[key][k]))
        for k in range(N_GROUPS)])
    group_height_measured[key] = np.array([
        float(evaluate_band_fit(*fit_band(group_real_rms[k], key), group_peak[key][k]))
        for k in range(N_GROUPS)])

height_r, height_p, height_r_measured = {}, {}, {}
for key in FIT_BANDS:
    result = pearsonr(group_median_f0, group_height[key])
    height_r[key], height_p[key] = result.statistic, result.pvalue
    height_r_measured[key] = pearsonr(group_median_f0, group_height_measured[key]).statistic
height_p_fdr = dict(zip(FIT_BANDS, false_discovery_control(
    np.array([height_p[key] for key in FIT_BANDS]), method="bh")))

fig, axes = plt.subplots(1, len(FIT_BANDS), figsize=(6.27 * len(FIT_BANDS) / 3, 2.0),
                         gridspec_kw=dict(wspace=0.45))
axes = np.atleast_1d(axes)
for ax, key in zip(axes, FIT_BANDS):
    low, high = FIT_BANDS[key]
    scatter_panel(ax, group_median_f0, group_height[key], group_hours,
                  height_r[key], height_p_fdr[key], xlabel="Group median F0 (Hz)",
                  ylabel="Peak coh. (excess)" if ax is axes[0] else None, headroom=0.75)
    ax.set_title(f"{low:g}-{high:g} Hz peak coherence", fontsize=ROW_TITLE_FONTSIZE)

stem = f"pitch_peak_height_{REFERENCE}"
fig.savefig(OUT_DIR / f"{stem}.pdf", bbox_inches="tight", pad_inches=0.3)
fig.savefig(OUT_DIR / f"{stem}.png", bbox_inches="tight", pad_inches=0.3, dpi=200)
plt.show()

print(f"{'group':>5s} {'median F0 (Hz)':>15s}"
      + "".join(f"{f'height {low:g}-{high:g}':>16s}" for low, high in FIT_BANDS.values()))
for k in range(N_GROUPS):
    print(f"{k:5d} {group_median_f0[k]:15.0f}"
          + "".join(f"{group_height[key][k]:16.4f}" for key in FIT_BANDS))
print()
for key, (low, high) in FIT_BANDS.items():
    print(f"{low:g}-{high:g} Hz peak height vs group median F0: r = {height_r[key]:+.2f}, "
          f"p = {height_p[key]:.3f}, p_FDR = {height_p_fdr[key]:.3f} (n = {N_GROUPS}); "
          f"on the measured spectrum r = {height_r_measured[key]:+.2f}")

## Supplementary figure -- the central cluster

Everything above takes the RMS over all 64 sensors. Figure 2 (notebook 2, panel (d) of
its combined figure) shows that above 30 Hz the tracking is focal over the midline, and
averages a 15-channel central cluster there instead: F1/Fz/F2, FC1/FCz/FC2, C1/Cz/C2,
CP1/CPz/CP2, P1/Pz/P2. Averaging all 64 dilutes a midline effect with channels that do
not carry it, so the pitch analysis is repeated on that cluster.

The groups are the same 8 as above. Each group's complex coherence is pooled per
channel, duration-weighted, before the magnitude is taken, then RMS-averaged over the
cluster, as notebook 2 does. The null is put through the same steps. This needs one more
pass over the per-stimulus files, since the cache above holds only the all-sensor RMS,
so it is cached on its own. Both questions are then asked again on the
null-corrected excess with the same band fits: where the peak is (as panels (c)-(d))
and how high it is (as the peak-height figure). There is one BH family over the bands
per row.

In [ ]:
from concurrent.futures import ThreadPoolExecutor

from coherence.pipeline import reference_channel_names

# Notebook 2's SUBBAND_CENTRAL_CLUSTER, its figure 2 panel (d).
CENTRAL_CLUSTER = ["F1", "Fz", "F2", "FC1", "FCz", "FC2", "C1", "Cz", "C2",
                   "CP1", "CPz", "CP2", "P1", "Pz", "P2"]
TEMPLATE_EEG = (Path(CONFIG["eeg_dir"])
                / "sub-001_-_ses-001_-_trial-001_-_cond-natural_-_audio01-audiobook_5_1_-_eeg.fif")
FILE_READERS = 4              # parallel file reads; each holds one ~655 MB null array


def channel_indices(reference, wanted):
    """Positions of `wanted` on the saved channel axis, which is the reference's EEG
    channel order (pipeline.load_eeg picks "eeg"). Missing channels are reported --
    FCz is dropped by its own reference."""
    names = reference_channel_names(TEMPLATE_EEG, reference)
    kept = [name for name in wanted if name in names]
    missing = sorted(set(wanted) - set(kept))
    if missing:
        print(f"central cluster: {len(kept)} of {len(wanted)} channels "
              f"({', '.join(missing)} not in the {reference} reference)")
    return [names.index(name) for name in kept], kept


def pitch_groups_on_channels(reference, channels, label):
    """The pitch groups' measured RMS and null-median RMS over `channels` only.

    The same groups, weights and order of operations as pool_pitch_groups: complex
    coherence pooled per channel, magnitude after, RMS over the channels. Cached under
    `label`, keyed on the stimuli, the groups and the channels.
    """
    directory = (resolve_reference_dirs(CONFIG, reference, ROOT)[1]
                 / "gammatone_null_by_stimulus")
    cache = (cache_dir(reference) /
             f"pitch_groups_{BAND[0]:g}-{BAND[1]:g}Hz_{N_GROUPS}groups"
             f"_{N_NULL_DRAWS}draws_{label}.npz")
    membership = [list(map(int, m)) for m in group_members]
    if cache.exists():
        with np.load(cache, allow_pickle=True) as saved:
            if ([str(n) for n in saved["stimuli"]] == list(stimulus_names)
                    and [list(map(int, m)) for m in saved["group_members"]] == membership
                    and list(saved["channels"]) == list(channels)):
                return saved["group_real_rms"], saved["group_null_median"]
        print(f"  {label}: cache differs, re-pooling")

    with np.load(directory / f"{stimulus_names[0]}_gammatone_null.npz") as first:
        in_band = (first["frequencies"] >= BAND[0]) & (first["frequencies"] <= BAND[1])

    def read(index):
        with np.load(directory / f"{stimulus_names[index]}_gammatone_null.npz") as saved:
            weight = float(saved["total_analysed_duration"])
            real = saved["gammatone_coherence"].squeeze(1)[in_band][:, channels]
            null = (saved["null_gammatone_coherence"][:N_NULL_DRAWS].squeeze(2)
                    [:, in_band][:, :, channels])
        return index, weight, real, null

    belongs = {index: k for k, members in enumerate(membership) for index in members}
    accumulators = [None] * N_GROUPS
    with ThreadPoolExecutor(FILE_READERS) as pool:
        for index, weight, real, null in tqdm(pool.map(read, range(len(stimulus_names))),
                                              total=len(stimulus_names), desc=label):
            k = belongs[index]
            if accumulators[k] is None:
                accumulators[k] = [np.zeros(real.shape, np.complex128),
                                   np.zeros(null.shape, np.complex64), 0.0]
            accumulators[k][0] += real * weight
            accumulators[k][1] += null * np.float32(weight)
            accumulators[k][2] += weight

    real_rms, null_median = [], []
    for real_sum, null_sum, weight_sum in accumulators:
        real_rms.append(np.sqrt((np.abs(real_sum / weight_sum) ** 2).mean(-1)))
        null_rms = np.sqrt((np.abs(null_sum / np.complex64(weight_sum)) ** 2).mean(-1))
        null_median.append(np.median(null_rms, axis=0))
    real_rms, null_median = np.array(real_rms), np.array(null_median)
    np.savez(cache, group_real_rms=real_rms, group_null_median=null_median,
             group_members=np.array([np.array(m) for m in membership], dtype=object),
             channels=np.array(channels), stimuli=np.array(stimulus_names))
    return real_rms, null_median


central_index, central_names = channel_indices(REFERENCE, CENTRAL_CLUSTER)
central_real_rms, central_null_median = pitch_groups_on_channels(REFERENCE, central_index,
                                                                 "central")
central_excess = central_real_rms - central_null_median
print(f"central cluster ({len(central_names)} channels): null floor "
      f"{(central_null_median / central_real_rms).mean() * 100:.0f}% of the observed RMS "
      f"over {BAND[0]:g}-{BAND[1]:g} Hz, against "
      f"{(group_null_median / group_real_rms).mean() * 100:.0f}% over all sensors")

In [ ]:
# Peak location and peak height on the central cluster, with the same band fits.
central_peak, central_height = {}, {}
for key in FIT_BANDS:
    results = [band_polynomial_peak(central_excess[k], key) for k in range(N_GROUPS)]
    central_peak[key] = np.array([peak for peak, _, _ in results])
    central_height[key] = np.array([
        float(evaluate_band_fit(*fit_band(central_excess[k], key), central_peak[key][k]))
        for k in range(N_GROUPS)])
    edge = sum(not interior for _, _, interior in results)
    print(f"{FIT_BANDS[key][0]:g}-{FIT_BANDS[key][1]:g} Hz on the central cluster: peak "
          f"{central_peak[key].mean():.1f} Hz (sd {central_peak[key].std():.1f}), "
          f"R2 {np.mean([r2 for _, r2, _ in results]):.2f}"
          + (f", {edge} of {N_GROUPS} pinned to a band edge" if edge else ""))

CENTRAL_ROWS = [("peak", central_peak, "Peak frequency (Hz)", "peak frequency", group_peak),
                ("height", central_height, "Peak coh. (excess)", "peak coherence", group_height)]
central_statistics = {}
fig, axes = plt.subplots(len(CENTRAL_ROWS), len(FIT_BANDS),
                         figsize=(6.27 * len(FIT_BANDS) / 3, 2.0 * len(CENTRAL_ROWS)),
                         gridspec_kw=dict(wspace=0.45, hspace=0.75), squeeze=False)
for row, (name, values, ylabel, title, all_sensors) in enumerate(CENTRAL_ROWS):
    raw = {key: pearsonr(group_median_f0, values[key]) for key in FIT_BANDS}
    corrected = dict(zip(FIT_BANDS, false_discovery_control(
        np.array([raw[key].pvalue for key in FIT_BANDS]), method="bh")))
    for column, key in enumerate(FIT_BANDS):
        low, high = FIT_BANDS[key]
        central_statistics[(name, key)] = dict(
            r=raw[key].statistic, p=raw[key].pvalue, p_fdr=corrected[key],
            r_all=pearsonr(group_median_f0, all_sensors[key]).statistic)
        ax = axes[row, column]
        scatter_panel(ax, group_median_f0, values[key], group_hours, raw[key].statistic,
                      corrected[key], xlabel="Group median F0 (Hz)",
                      ylabel=ylabel if column == 0 else None, headroom=0.75)
        ax.set_title(f"{low:g}-{high:g} Hz {title}", fontsize=ROW_TITLE_FONTSIZE)

stem = f"pitch_central_cluster_{REFERENCE}"
fig.savefig(OUT_DIR / f"{stem}.pdf", bbox_inches="tight", pad_inches=0.3)
fig.savefig(OUT_DIR / f"{stem}.png", bbox_inches="tight", pad_inches=0.3, dpi=200)
plt.show()

print(f"{'group':>5s} {'median F0 (Hz)':>15s}"
      + "".join(f"{f'peak {low:g}-{high:g}':>14s}{f'height {low:g}-{high:g}':>16s}"
                for low, high in FIT_BANDS.values()))
for k in range(N_GROUPS):
    print(f"{k:5d} {group_median_f0[k]:15.0f}"
          + "".join(f"{central_peak[key][k]:14.1f}{central_height[key][k]:16.4f}"
                    for key in FIT_BANDS))
print()
for (name, key), stats in central_statistics.items():
    low, high = FIT_BANDS[key]
    print(f"central, {low:g}-{high:g} Hz peak {'frequency' if name == 'peak' else 'height'} "
          f"vs group median F0: r = {stats['r']:+.2f}, p = {stats['p']:.3f}, "
          f"p_FDR = {stats['p_fdr']:.3f} (n = {N_GROUPS}); all sensors r = {stats['r_all']:+.2f}")